<a href="https://colab.research.google.com/github/JMan003/RAG-LLM/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages
!pip install -q transformers sentence_transformers llama-index langchain pyngrok flask torch faiss-cpu pymupdf

In [ ]:
!pip install llama-index-llms-huggingface

In [ ]:
!pip install -q langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.7 MB/s eta 0:00:00


In [ ]:
!pip install -q llama-index-embeddings-langchain

In [ ]:
import os
import torch
import logging
import sys
from tqdm.notebook import tqdm
from flask import Flask, request, jsonify
from pyngrok import ngrok
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core.prompts.prompts import SimpleInputPrompt
from langchain.embeddings.huggingface import HuggingFaceEmbeddings
import fitz  # PyMuPDF
from transformers import AutoModelForCausalLM, AutoTokenizer

# Set up logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger("pypdf").setLevel(logging.ERROR)

# Create Flask app
app = Flask(__name__)

# Create data directory if it doesn't exist
!mkdir -p data

# Extract text from PDF using PyMuPDF
def extract_text_from_pdf(pdf_path):
    try:
        text = ""
        with fitz.open(pdf_path) as doc:
            for page in doc:
                text += page.get_text()
        return text
    except Exception as e:
        print(f"Error extracting text from {pdf_path}: {str(e)}")
        return ""

# Function to load and chunk documents with progress bar
def load_and_process_documents():
    print("Loading documents...")
    documents = []
    data_dir = './data'

    # Count files first
    files = [f for f in os.listdir(data_dir) if os.path.isfile(os.path.join(data_dir, f))]

    for filename in tqdm(files, desc="Processing files"):
        file_path = os.path.join(data_dir, filename)
        try:
            if filename.lower().endswith('.pdf'):
                text = extract_text_from_pdf(file_path)
                if not text.strip():
                    print(f"Warning: No text extracted from {filename}")
                    continue
            elif filename.lower().endswith(('.txt', '.md')):
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    text = f.read()
            else:
                continue

            documents.append(Document(text=text, metadata={"source": filename}))
            print(f"Successfully loaded {filename} ({len(text)} characters)")
        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")

    return documents

# Create index from documents with aggressive chunking and progress reporting
def create_vector_index(documents, embed_model):
    print(f"Creating vector index from {len(documents)} documents...")

    # Use a more aggressive text splitter to create smaller chunks
    text_splitter = SentenceSplitter(
        chunk_size=256,  # Smaller chunks
        chunk_overlap=20,
        paragraph_separator="\n\n",
        separator=" "
    )

    print("Splitting documents into chunks...")
    nodes = []
    for doc in tqdm(documents, desc="Chunking documents"):
        doc_nodes = text_splitter.get_nodes_from_documents([doc])
        nodes.extend(doc_nodes)

    print(f"Created {len(nodes)} text chunks")

    # Create index from nodes to better manage memory
    print("Building vector index...")
    index = VectorStoreIndex(nodes)

    return index

# Function to truncate text to a safe length
def truncate_text(text, max_words=200):
    """Truncate text to approximately max_words"""
    words = text.split()
    if len(words) <= max_words:
        return text
    return ' '.join(words[:max_words]) + "..."

# Format response in LEAP structure with length limits
def format_leap_response(query, raw_response):
    """Format the response according to the L.E.A.P. structure with length limits"""

    # Check if response already has LEAP structure and is reasonably sized
    if "Legal Issue" in raw_response and "Action Steps" in raw_response:
        words = raw_response.split()
        if len(words) <= 400:  # If already in LEAP format and not too long
            return raw_response

    # Simple assessment of query type to customize response
    query_lower = query.lower()

    # Identify potential legal issues based on keywords
    if any(word in query_lower for word in ["accident", "crash", "collision"]):
        issue = "a motor vehicle accident and potential liability under road safety laws"
    elif any(word in query_lower for word in ["property", "land", "tenant", "owner"]):
        issue = "a property dispute or real estate matter"
    elif any(word in query_lower for word in ["marriage", "divorce", "custody"]):
        issue = "a family law matter"
    elif any(word in query_lower for word in ["job", "salary", "employer", "fired"]):
        issue = "an employment law issue"
    else:
        issue = "a potential legal concern requiring professional advice"

    # Truncate raw response to a reasonable length
    truncated_response = truncate_text(str(raw_response), 150)

    # Format the response with controlled length for each section
    formatted_response = f"""
**Legal Issue**:
This query concerns {issue}.

**Explanation of Law**:
{truncated_response}

**Action Steps**:
1. Consult with a qualified legal professional specializing in this area of law
2. Gather all relevant documentation and evidence related to your case
3. Consider filing appropriate paperwork with relevant authorities

**Practical Guidance**:
Maintain thorough documentation of all events, communications, and expenses related to this matter.
"""
    return formatted_response

# Main function to set up the system
def setup_rag_system():
    print("Setting up RAG system...")

    # Define the Lawracle system prompt - shortened for token efficiency
    system_prompt = """
You are Lawracle, an AI legal assistant specializing in Indian law. Provide expert legal advice focused on legal actions, references to laws, and clear guidance. Use the L.E.A.P. structure in all responses:
1. Legal Issue: Identify the core legal problem concisely.
2. Explanation of Law: Provide relevant sections and legal concepts.
3. Action Steps: List practical steps the client should take.
4. Practical Guidance: Offer tips to strengthen the client's case.
Keep responses clear, professional, and concise, under 400 words.
"""

    # Define query wrapper prompt
    query_wrapper_prompt = SimpleInputPrompt("<|USER|>{query_str}<|ASSISTANT|>")

    # Use a model with reduced token output
    llm = "meta-llama/Llama-2-7b-chat-hf"  # Change if using another model
    tokenizer = AutoTokenizer.from_pretrained(llm)
    model = AutoModelForCausalLM.from_pretrained(llm)

    # Use a smaller, faster embedding model
    embed_model = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",  # Much smaller embedding model
        model_kwargs={"device": "cpu"}
    )

    # Configure settings
    Settings.llm = llm
    Settings.embed_model = embed_model
    Settings.chunk_size = 256
    Settings.chunk_overlap = 20

    # Load and process documents
    documents = load_and_process_documents()

    if not documents:
        print("No documents were loaded. Creating a sample document...")
        sample_text = "Sample legal text about Motor Vehicles Act Section 134."
        documents = [Document(text=sample_text, metadata={"source": "sample.txt"})]

    # Create index
    index = create_vector_index(documents, embed_model)

    # Create query engine with response length limits
    query_engine = index.as_query_engine(
        response_mode="compact",
        similarity_top_k=2,  # Reduced from 3 to limit context size
    )

    return query_engine

# API route for queries with error handling for length issues
@app.route('/query', methods=['POST'])
def process_query():
    data = request.json
    if not data or 'query' not in data:
        return jsonify({"error": "No query provided"}), 400

    query = data['query']
    print(f"Processing query: {query}")

    try:
        # Get raw response from query engine - handle possible exceptions
        try:
            raw_response = query_engine.query(query)
        except ValueError as e:
            if "exceed the model's predefined maximum length" in str(e):
                print("Warning: Response exceeded length limit. Using truncated context.")
                raw_response = "The retrieved information was too lengthy. Please ask a more specific question."
            else:
                raise e

        # Format in LEAP structure with length control
        formatted_response = format_leap_response(query, str(raw_response))

        return jsonify({"response": formatted_response})
    except Exception as e:
        print(f"Error: {str(e)}")
        # Fallback response if there's an error
        fallback_response = format_leap_response(query, "Unable to retrieve specific information for your query.")
        return jsonify({"response": fallback_response})

# Main execution
if __name__ == '__main__':
    try:
        print("Initializing RAG system...")
        query_engine = setup_rag_system()

        # Set up ngrok
        print("\nTo create a public URL, you'll need an ngrok auth token.")
        print("Get one for free at https://dashboard.ngrok.com")
        ngrok_auth_token = input("Enter your ngrok auth token (press Enter to skip): ")

        if ngrok_auth_token:
            ngrok.set_auth_token(ngrok_auth_token)
            public_url = ngrok.connect(5000)
            print(f"API is publicly available at: {public_url}/query")

        # Start the Flask app
        app.run(host='0.0.0.0', port=5000)
    except Exception as e:
        print(f"Fatal error: {str(e)}")

ModuleNotFoundError: No module named 'pyngrok'